# Açıklanabilir Yapay Zekâ (XAI) – SHAP & LIME Uygulamaları
Bu notebook, **satır içi Türkçe açıklamalar** ile aşağıdaki adımları içerir:
1. Breast Cancer veri seti + XGBoost modeli eğitimi
2. SHAP ile küresel öznitelik etkisi analizi (feature importance, beeswarm)
3. LIME ile bireysel tahmin açıklaması
4. Yanlış sınıflandırılan örnekler ve etik tartışma soruları
5. Mini görevler (XAI yorumlama oturumu)


In [ ]:
# Kütüphaneler
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import shap                       # SHAP açıklamaları
import lime.lime_tabular as lime   # LIME açıklamaları
from xgboost import XGBClassifier  # Model

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

## 1. Veri Seti ve Model Hazırlığı

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)

# Eğitim & test bölme
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Ölçekleme
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# XGBoost sınıflandırıcı
model = XGBClassifier(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)
model.fit(X_train_scaled, y_train)

# Performans
y_pred = model.predict(X_test_scaled)
print("Test Doğruluğu:", accuracy_score(y_test, y_pred).round(3))


## 2. SHAP ile Küresel Öznitelik Etkisi

In [ ]:
# SHAP TreeExplainer
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test_scaled)

# Özellik önem grafiği
shap.summary_plot(shap_values, features=X_test, feature_names=data.feature_names)


In [ ]:
# Beeswarm grafiği – her örnek için SHAP katkıları
shap.summary_plot(shap_values, features=X_test, feature_names=data.feature_names, plot_type='beeswarm')


## 3. LIME ile Bireysel Tahmin Açıklaması

In [ ]:
explainer_lime = lime.LimeTabularExplainer(
    X_train_scaled,
    feature_names=data.feature_names,
    class_names=data.target_names,
    discretize_continuous=True
)

idx = 5  # keyfi bir test örneği
exp = explainer_lime.explain_instance(
    X_test_scaled[idx],
    model.predict_proba,
    num_features=10
)
exp.show_in_notebook(show_table=True)

## 4. Yanlış Sınıflandırılan Örnekler ve Etik Tartışma

In [ ]:
wrong_idx = np.where(y_pred != y_test)[0]
print(f"Yanlış sınıflandırılan örnek sayısı: {len(wrong_idx)}")

# İlk yanlış tahmini incele
if len(wrong_idx):
    i = wrong_idx[0]
    print(f"Gerçek sınıf: {data.target_names[y_test.iloc[i]]}, Tahmin: {data.target_names[y_pred[i]]}")
    exp_wrong = explainer_lime.explain_instance(
        X_test_scaled[i],
        model.predict_proba,
        num_features=10
    )
    exp_wrong.show_in_notebook(show_table=True)
else:
    print("Model tüm test örneklerini doğru sınıflandırdı!")


### 4.1 Etik Tartışma Soruları
1. **Yanlış Negatiflerin Maliyeti**: Meme kanseri teşhisinde hatalı negatif (hasta ⇒ sağlıklı) kararı neleri etkiler? Klinik ve psikolojik sonuçları düşünün.
2. **Özellik Açıklamaları**: SHAP/LIME, modelin hangi tıbbi ölçümlere dayandığını gösteriyor. Bu bilgiler hekim kararlarını destekler mi, yoksa yanıltabilir mi?
3. **Veri Önyargısı**: Veri setindeki örnek dağılımı (% malign / benign) gerçek hayattaki dağılımı temsil etmiyor olabilir. Model adaletsiz sonuçlar üretebilir mi?
4. **Gizlilik**: Hasta verileri ile çalışırken SHAP/LIME çıktıları kişisel bilgileri ifşa eder mi? Nasıl önlem alınabilir?


## 📝 Mini Görevler
1. **SHAP Dependence Plot**: `mean radius` özelliği için SHAP bağımlılık grafiğini çizin.
2. **Farklı Model**: RandomForest modeli eğitip SHAP/LIME sonuçlarını karşılaştırın.
3. **Top 5 Yanlış**: En yüksek olasılıkla yanlış tahmin edilen 5 örneği bulun ve açıklamalarını analiz edin.
4. **Fairness Check** *(opsiyonel)*: Veri setine sentetik “yaş” değişkeni ekleyip yaş gruplarına göre SHAP değerlerini inceleyin.
5. **Rapor Yazımı**: SHAP & LIME bulgularını, etik riskleri ve çözüm önerilerini içeren 1 sayfalık kısa rapor hazırlayın.